# Avance 2 - Integración de API Yelp
## Para esto, vamos a conectarnos a la API de Yelp usando la API Key brindada.
### ¿Qué haremos?
#### * Conectar con Yelp Fusion API para obtener datos de negocios locales (nombre, categoría, calificación, reseñas, etc.) según ciudad seleccionada en el Avance 1
#### * Integrar las variables del nuevo dataset con el análisis existente (ej.: generar funciones que combinen datos de ambos DataFrames, como ofrecer recomendaciones).

In [13]:
# Avance 2: API de Yelp - Versión mejorada para más resultados
import pandas as pd
import requests
import json
import time

# Tu API Key de Yelp
api_key = 'FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'

# Configurar headers para la API
headers = {
    'Authorization': f'Bearer {api_key}',
    'accept': 'application/json'
}

# Función mejorada para buscar más negocios
def buscar_negocios_yelp_completo(termino='restaurants', ubicacion='Chicago', total_limite=200):
    url = 'https://api.yelp.com/v3/businesses/search'
    todos_negocios = []
    offset = 0
    limite_por_request = 50  # Máximo permitido por Yelp
    
    print(f"Buscando hasta {total_limite} restaurantes en {ubicacion}...")
    
    while offset < total_limite:
        parametros = {
            'term': termino,
            'location': ubicacion,
            'limit': limite_por_request,
            'offset': offset,
            'sort_by': 'best_match'
        }

        try:
            response = requests.get(url, headers=headers, params=parametros)
            response.raise_for_status()
            
            datos = response.json()
            negocios = datos.get('businesses', [])
            
            if not negocios:
                break  # No hay más resultados
                
            todos_negocios.extend(negocios)
            print(f"Obtenidos {len(negocios)} restaurantes (offset: {offset})")
            
            # Pequeña pausa para no saturar la API
            time.sleep(0.5)
            
            offset += len(negocios)
            
            # Si ya tenemos suficientes o no hay más, salir
            if len(negocios) < limite_por_request or offset >= total_limite:
                break
                
        except requests.exceptions.RequestException as e:
            print(f"Error en la solicitud (offset {offset}): {e}")
            break
        except json.JSONDecodeError as e:
            print(f"Error decodificando JSON: {e}")
            break
    
    print(f"Total obtenido: {len(todos_negocios)} restaurantes")
    return todos_negocios

# Obtener datos de restaurantes en Chicago (hasta 200)
restaurantes_chicago = buscar_negocios_yelp_completo(termino='restaurants', ubicacion='Chicago', total_limite=200)

if restaurantes_chicago:
    print("\nTodos los campos disponibles del primer restaurante:")
    primer_negocio = restaurantes_chicago[0]
    for key in primer_negocio.keys():
        print(f"- {key}")

Buscando hasta 200 restaurantes en Chicago...
Obtenidos 50 restaurantes (offset: 0)
Obtenidos 50 restaurantes (offset: 50)
Obtenidos 50 restaurantes (offset: 100)
Obtenidos 50 restaurantes (offset: 150)
Total obtenido: 200 restaurantes

Todos los campos disponibles del primer restaurante:
- id
- alias
- name
- image_url
- is_closed
- url
- review_count
- categories
- rating
- coordinates
- transactions
- price
- location
- phone
- display_phone
- distance


In [14]:
# Analizamos la estructura de datos recibida
if restaurantes_chicago:
    print("\n" + "="*60)
    print("ANÁLISIS DETALLADO DE TIPOS DE DATOS")
    print("="*60)
    
    primer_negocio = restaurantes_chicago[0]
    print(f"Campos totales en cada negocio: {len(primer_negocio)}")
    
    print(f"\n{'CAMPO':<25} {'TIPO':<20} {'EJEMPLO':<30}")
    print("-" * 75)
    
    # Analizar cada campo del primer negocio
    for key, value in primer_negocio.items():
        # Determinamos el tipo específico
        if value is None:
            tipo_especifico = "Nulo"
        elif isinstance(value, bool):
            tipo_especifico = "Booleano"
        elif isinstance(value, int):
            tipo_especifico = "Entero"
        elif isinstance(value, float):
            tipo_especifico = "Decimal"
        elif isinstance(value, str):
            if value.isdigit():
                tipo_especifico = "Texto numerico"
            else:
                tipo_especifico = "Texto"
        elif isinstance(value, dict):
            tipo_especifico = f"Objeto ({len(value)} campos)"
        elif isinstance(value, list):
            tipo_especifico = f"Lista ({len(value)} elementos)"
            if value:
                primer_elemento = value[0]
                tipo_especifico += f" de {type(primer_elemento).__name__}"
        else:
            tipo_especifico = type(value).__name__
        
        # Valor de ejemplo recortado
        ejemplo = str(value)
        if len(ejemplo) > 28:
            ejemplo = ejemplo[:25] + "..."
        
        print(f"{key:<25} {tipo_especifico:<20} {ejemplo:<30}")


ANÁLISIS DETALLADO DE TIPOS DE DATOS
Campos totales en cada negocio: 16

CAMPO                     TIPO                 EJEMPLO                       
---------------------------------------------------------------------------
id                        Texto                qjnpkS8yZO8xcyEIy5OU9A        
alias                     Texto                girl-and-the-goat-chicago     
name                      Texto                Girl & The Goat               
image_url                 Texto                https://s3-media0.fl.yelp...  
is_closed                 Booleano             False                         
url                       Texto                https://www.yelp.com/biz/...  
review_count              Entero               10510                         
categories                Lista (3 elementos) de dict [{'alias': 'newamerican',...  
rating                    Decimal              4.4                           
coordinates               Objeto (2 campos)    {'latitude': 41.

In [15]:
# Crear DataFrame con los datos relevantes
print("\n" + "="*60)
print("CREACION DEL DATAFRAME")
print("="*60)

datos_restaurantes = []

for negocio in restaurantes_chicago:
    # Extraer categorias como texto
    categorias = ', '.join([cat['title'] for cat in negocio.get('categories', [])])
    
    # Extraer direccion completa
    direccion = ', '.join(negocio['location'].get('display_address', []))
    
    datos_restaurantes.append({
        'id_yelp': negocio.get('id'),
        'nombre_negocio': negocio.get('name'),
        'alias': negocio.get('alias'),
        'rating': negocio.get('rating'),
        'reseñas': negocio.get('review_count'),
        'precio': negocio.get('price', 'No especificado'),
        'categorias': categorias,
        'direccion': direccion,
        'ciudad': negocio['location'].get('city'),
        'estado': negocio['location'].get('state'),
        'codigo_postal': negocio['location'].get('zip_code'),
        'pais': negocio['location'].get('country'),
        'telefono': negocio.get('phone'),
        'telefono_display': negocio.get('display_phone'),
        'esta_cerrado': negocio.get('is_closed', False),
        'url_yelp': negocio.get('url'),
        'image_url': negocio.get('image_url'),
        'distancia': negocio.get('distance')  # En metros
    })

# Crear DataFrame
df_restaurantes = pd.DataFrame(datos_restaurantes)

print(f"Dimensiones del DataFrame: {df_restaurantes.shape}")
print(f"Columnas: {list(df_restaurantes.columns)}")

# Mostrar información basica del DataFrame
print("\n Primeras 5 filas del DataFrame:")
print(df_restaurantes.head())

print("\n Información del DataFrame:")
print(df_restaurantes.info())

print("\n Estadísticas basicas de columnas numericas:")
print(df_restaurantes[['rating', 'reseñas', 'distancia']].describe())

# Guardar datos en CSV en la carpeta Archivos CSV
ruta_guardado = "../Archivos_CSV/restaurantes_chicago_yelp.csv"
df_restaurantes.to_csv(ruta_guardado, index=False)
print(f"\nDatos guardados en: {ruta_guardado}")



CREACION DEL DATAFRAME
Dimensiones del DataFrame: (200, 18)
Columnas: ['id_yelp', 'nombre_negocio', 'alias', 'rating', 'reseñas', 'precio', 'categorias', 'direccion', 'ciudad', 'estado', 'codigo_postal', 'pais', 'telefono', 'telefono_display', 'esta_cerrado', 'url_yelp', 'image_url', 'distancia']

 Primeras 5 filas del DataFrame:
                  id_yelp   nombre_negocio                      alias  rating  \
0  qjnpkS8yZO8xcyEIy5OU9A  Girl & The Goat  girl-and-the-goat-chicago     4.4   
1  boE4Ahsssqic7o5wQLI04w   The Purple Pig     the-purple-pig-chicago     4.3   
2  gzhkdb6YoiFm5s3vriG1AA           Gretel             gretel-chicago     4.5   
3  riT822EnU7y_5eCuJsd9sA  Cindy's Rooftop     cindys-rooftop-chicago     4.1   
4  cgIuo3geaxw1Sfhbqm5qTA        Alla Vita        alla-vita-chicago-4     4.4   

   reseñas precio                                       categorias  \
0    10510    $$$                     New American, Bars, Bakeries   
1     8856    $$$  Tapas/Small Plates, M

### Verificaciones rapidas del archivo

In [16]:
# Verificamos: Realmente solo tenemos Chicago?
ciudades_unicas = df_restaurantes['ciudad'].unique()
print(f"Ciudades encontradas: {len(ciudades_unicas)} - {list(ciudades_unicas)}")

# Validacion específica para Chicago
if len(ciudades_unicas) == 1 and ciudades_unicas[0] == 'Chicago':
    print("VALIDACION EXITOSA: Todos los restaurantes estan en Chicago")
else:
    print("ALERTA: Se encontraron restaurantes fuera de Chicago")
    print("Esto puede indicar que:")
    print(" -La API incluyo áreas metropolitanas cercanas")
    print(" -Hay errores en los datos de ubicación")
    
    # Mostrar distribución exacta por ciudad
    print("\n Distribucion por ciudad:")
    distribucion_ciudades = df_restaurantes['ciudad'].value_counts()
    for ciudad, count in distribucion_ciudades.items():
        print(f"   - {ciudad}: {count} restaurantes ({count/len(df_restaurantes)*100:.1f}%)")

    # IDENTIFICAR Y ELIMINAR RESTAURANTES FUERA DE CHICAGO
    print(f"\n Limpiando datos: Eliminando restaurantes fuera de Chicago...")

    # Guardar registro de lo que se elimina
    restaurantes_fuera_chicago = df_restaurantes[df_restaurantes['ciudad'] != 'Chicago']
    print(f"   Restaurantes a eliminar ({len(restaurantes_fuera_chicago)}):")
    for _, restaurante in restaurantes_fuera_chicago.iterrows():
        print(f"     - {restaurante['nombre_negocio']} ({restaurante['ciudad']})")
    
    # Filtrar para mantener solo Chicago
    registros_antes = len(df_restaurantes)
    df_restaurantes = df_restaurantes[df_restaurantes['ciudad'] == 'Chicago'].copy()
    registros_despues = len(df_restaurantes)
    
    print(f"\n Limpieza completada:")
    print(f" -Registros antes: {registros_antes}")
    print(f" -Registros después: {registros_despues}")
    print(f" -Registros eliminados: {registros_antes - registros_despues}")
    
    # Verificacion final
    ciudades_finales = df_restaurantes['ciudad'].unique()
    print(f" -Ciudades finales: {list(ciudades_finales)}")

Ciudades encontradas: 2 - ['Chicago', 'Lincoln Park']
ALERTA: Se encontraron restaurantes fuera de Chicago
Esto puede indicar que:
 -La API incluyo áreas metropolitanas cercanas
 -Hay errores en los datos de ubicación

 Distribucion por ciudad:
   - Chicago: 199 restaurantes (99.5%)
   - Lincoln Park: 1 restaurantes (0.5%)

 Limpiando datos: Eliminando restaurantes fuera de Chicago...
   Restaurantes a eliminar (1):
     - DeNuccis (Lincoln Park)

 Limpieza completada:
 -Registros antes: 200
 -Registros después: 199
 -Registros eliminados: 1
 -Ciudades finales: ['Chicago']


### Continuamos con la verificacion de datos, para buscar datos que nos sirvan de interseccion con el archivo utilizado en el avance 1

In [17]:
# Verificacion general
print("\n RESUMEN GENERAL:")
print(f"Total de restaurantes obtenidos: {len(df_restaurantes)}")
print(f"Rango de ratings: {df_restaurantes['rating'].min()} - {df_restaurantes['rating'].max()}")
print(f"Total de reseñas analizadas: {df_restaurantes['reseñas'].sum():,}")
print(f"Niveles de precio encontrados: {sorted(df_restaurantes['precio'].unique())}")
print(f"Dataset con {df_restaurantes.shape[1]} columnas y {df_restaurantes.shape[0]} filas")

# Verificacion de valores nulos
print("\n VERIFICACION DE VALORES NULOS:")
hay_nulos = df_restaurantes.isnull().sum().any()

if hay_nulos:
    print("Sí, hay valores nulos en el dataset")
    print("Detalle por columna:")
    nulos_por_columna = df_restaurantes.isnull().sum()
    # Mostrar solo columnas con nulos
    nulos_por_columna = nulos_por_columna[nulos_por_columna > 0]
    for columna, cantidad in nulos_por_columna.items():
        print(f"   - {columna}: {cantidad} nulos ({cantidad/len(df_restaurantes)*100:.1f}%)")
else:
    print("No hay valores nulos en el dataset!")

# Verificacion de datos unicos
print("\n ANALISIS DE DATOS _UNICOS:")
print(f" -Ciudades unicas: {df_restaurantes['ciudad'].nunique()} - {list(df_restaurantes['ciudad'].unique())}")
print(f" -Categorias unicas: {df_restaurantes['categorias'].nunique()}")
print(f" -Niveles de precio: {df_restaurantes['precio'].nunique()}")

# Estadisticas adicionales
print("\n ESTADISTICAS ADICIONALES:")
print(f" -Rating promedio: {df_restaurantes['rating'].mean():.2f}")
print(f" -Reseñas promedio por restaurante: {df_restaurantes['reseñas'].mean():.1f}")
print(f" -Restaurante con más reseñas: {df_restaurantes.loc[df_restaurantes['reseñas'].idxmax(), 'nombre_negocio']} ({df_restaurantes['reseñas'].max()} reseñas)")


# Restaurantes cerrados vs abiertos
print("\n ESTADO DE RESTAURANTES:")
cerrados = df_restaurantes['esta_cerrado'].sum()
abiertos = len(df_restaurantes) - cerrados
print(f" Abiertos: {abiertos} ({abiertos/len(df_restaurantes)*100:.1f}%)")
print(f" Cerrados: {cerrados} ({cerrados/len(df_restaurantes)*100:.1f}%)")


 RESUMEN GENERAL:
Total de restaurantes obtenidos: 199
Rango de ratings: 3.8 - 5.0
Total de reseñas analizadas: 150,275
Niveles de precio encontrados: ['$$', '$$$', '$$$$', 'No especificado']
Dataset con 18 columnas y 199 filas

 VERIFICACION DE VALORES NULOS:
No hay valores nulos en el dataset!

 ANALISIS DE DATOS _UNICOS:
 -Ciudades unicas: 1 - ['Chicago']
 -Categorias unicas: 177
 -Niveles de precio: 4

 ESTADISTICAS ADICIONALES:
 -Rating promedio: 4.42
 -Reseñas promedio por restaurante: 755.2
 -Restaurante con más reseñas: Girl & The Goat (10510 reseñas)

 ESTADO DE RESTAURANTES:
 Abiertos: 199 (100.0%)
 Cerrados: 0 (0.0%)


## Procedemos a buscar puntos de union entre los archivos

In [18]:
#Para eso procedemos:

# 1. Cargando los 2 archivos desde la carpeta Archivos CSV
df_clientes = pd.read_csv("../Archivos_CSV/clientes_chicago_procesado.csv")
df_restaurantes = pd.read_csv("../Archivos_CSV/restaurantes_chicago_yelp.csv")

print(f"Clientes dataset: {df_clientes.shape}")
print(f"Restaurantes dataset: {df_restaurantes.shape}")

# 2. Revisemos si las categorias de Yelp se repiten, para buscar puntos de union con las preferencias alimenticias de los clientes:
print("Categorias de Restaurantes en Yelp")
print("="*60)

# Extraer todas las categorias individuales de Yelp
todas_categorias_yelp = []
for categorias in df_restaurantes['categorias']:
    if pd.notna(categorias):
        todas_categorias_yelp.extend([cat.strip() for cat in categorias.split(',')])

conteo_categorias_yelp = pd.Series(todas_categorias_yelp).value_counts()
print("Todas las categorias de Yelp y su frecuencia:")
print("-" * 60)
for categoria, count in conteo_categorias_yelp.items():
    print(f"   - {categoria}: {count} restaurantes")

print(f"\n Total de categorias unicas en Yelp: {len(conteo_categorias_yelp)}")
print(f" Total de apariciones de categorías: {conteo_categorias_yelp.sum()}")
print(f" Categoria más comun: {conteo_categorias_yelp.index[0]} ({conteo_categorias_yelp.iloc[0]} restaurantes)")

Clientes dataset: (5069, 16)
Restaurantes dataset: (200, 18)
Categorias de Restaurantes en Yelp
Todas las categorias de Yelp y su frecuencia:
------------------------------------------------------------
   - Cocktail Bars: 48 restaurantes
   - New American: 37 restaurantes
   - Italian: 28 restaurantes
   - Breakfast & Brunch: 16 restaurantes
   - Mediterranean: 16 restaurantes
   - Bars: 15 restaurantes
   - Seafood: 14 restaurantes
   - Wine Bars: 13 restaurantes
   - American: 13 restaurantes
   - Steakhouses: 12 restaurantes
   - Asian Fusion: 12 restaurantes
   - Korean: 10 restaurantes
   - Desserts: 10 restaurantes
   - Salad: 9 restaurantes
   - Mexican: 8 restaurantes
   - Indian: 7 restaurantes
   - Chinese: 7 restaurantes
   - French: 7 restaurantes
   - Burgers: 6 restaurantes
   - Tapas/Small Plates: 6 restaurantes
   - Cafes: 6 restaurantes
   - Japanese: 6 restaurantes
   - Beer Bar: 6 restaurantes
   - Ramen: 6 restaurantes
   - Barbeque: 5 restaurantes
   - Latin Ameri

In [19]:
# Veamos la cantidad de categorias por restaurante
print("Cantidad de Catgorias por Restarante")

categorias_por_restaurante = df_restaurantes['categorias'].apply(
    lambda x: len(x.split(',')) if pd.notna(x) else 0
)

# Contar cuantos restaurantes tienen X categorias
conteo = categorias_por_restaurante.value_counts().sort_index()

for num_cat, cantidad in conteo.items():
    print(f"{cantidad:2d} restaurantes tienen {num_cat}")

print(f"\nTotal: {len(df_restaurantes)} restaurantes analizados")

Cantidad de Catgorias por Restarante
45 restaurantes tienen 1
49 restaurantes tienen 2
101 restaurantes tienen 3
 5 restaurantes tienen 4

Total: 200 restaurantes analizados


In [20]:
# Bien, sigamos con clasificar restaurantes, para generar Match o recomendaciones a la base de clientes
print("="*60)
print("CLASIFICACION DE RESTAURANTES POR PREFERENCIAS")
print("="*60)

# Definimos los siguientes Match entre Preferencias alimenticias y Categorias de restaurantes:
clasificacion_categorias = {
    'Carnes': [
        'Steakhouses', 'Barbeque', 'Chicken Wings', 'Chicken Shop', 
        'Argentine', 'Comfort Food', 'Southern', 'Cajun/Creole', 'BBQ',
        'Steak', 'Meat', 'Grill', 'Barbecue'
    ],
    
    'Vegetariano_Vegano': [
        'Vegetarian', 'Vegan', 'Salad', 'Falafel', 'Lebanese', 
        'Middle Eastern', 'Turkish', 'Healthy', 'Vegetarian',
        'Vegan', 'Plant Based', 'Garden'
    ],
    
    'Mariscos_Pescado': [
        'Seafood', 'Sushi Bars', 'Japanese', 'Ramen', 'Poke',
        'Fish & Chips', 'Hot Pot', 'Sushi', 'Fish', 'Seafood',
        'Poke Bowls', 'Sashimi'
    ],
    
    'Consume_licor': [
        'Cocktail Bars', 'Bars', 'Wine Bars', 'Beer Bar', 'Wine & Spirits',
        'Beer', 'Pubs', 'Speakeasies', 'Whiskey Bars', 'Brewpubs',
        'Breweries', 'Lounges', 'Gastropubs', 'Bar', 'Pub', 'Wine',
        'Cocktail', 'Whiskey', 'Brewery', 'Gastropub'
    ]
}

# Clasificamos:
def clasificar_restaurante(categorias_str):
    """Clasifica un restaurante en una o más categorías"""
    if pd.isna(categorias_str):
        return ['Otro']
    
    categorias = [cat.strip() for cat in categorias_str.split(',')]
    clasificaciones = []
    
    for cat in categorias:
        for grupo, palabras_clave in clasificacion_categorias.items():
            if any(palabra.lower() in cat.lower() for palabra in palabras_clave):
                if grupo not in clasificaciones:
                    clasificaciones.append(grupo)
    
    # Si no coincide con ninguna categoria especifica, es "Otro"
    if not clasificaciones:
        clasificaciones = ['Otro']
    
    return clasificaciones

# Aplicar clasificacion
df_restaurantes['clasificacion'] = df_restaurantes['categorias'].apply(clasificar_restaurante)

# Sumamos columnas de clasificacion al DataFrame de restaurantes
print("Agregando columnas...")

# Función para determinar preferencia alimenticia principal
def obtener_preferencia_alimenticia(clasificaciones):
    """Obtiene la preferencia alimenticia principal del restaurante"""
    # Prioridad: excluir Consume_licor para preferencias alimenticias
    preferencias_alimenticias = [cat for cat in clasificaciones if cat != 'Consume_licor']
    
    if not preferencias_alimenticias:
        return 'Otro'
    elif len(preferencias_alimenticias) == 1:
        return preferencias_alimenticias[0]
    else:
        # Si tiene multiples, priorizar en este orden
        if 'Carnes' in preferencias_alimenticias:
            return 'Carnes'
        elif 'Mariscos_Pescado' in preferencias_alimenticias:
            return 'Mariscos_Pescado'
        elif 'Vegetariano_Vegano' in preferencias_alimenticias:
            return 'Vegetariano_Vegano'
        else:
            return 'Otro'

# Funcion para determinar si consume licor
def obtener_consume_licor(clasificaciones):
    """Determina si el restaurante está en la categoría de consume_licor"""
    return 'Si' if 'Consume_licor' in clasificaciones else 'No'

# Aplicar las nuevas columnas
df_restaurantes['preferencias_alimenticias'] = df_restaurantes['clasificacion'].apply(obtener_preferencia_alimenticia)
df_restaurantes['consume_licor'] = df_restaurantes['clasificacion'].apply(obtener_consume_licor)

# Veamos el resultado de la clasificacion:
print("CLASIFICACION COMPLETADA, LOS RESULTADOS SON:")
print("-" * 60)

print("DISTRIBUCION DE PREFERENCIAS ALIMENTICIAS:")
pref_alimenticias = df_restaurantes['preferencias_alimenticias'].value_counts()
for pref, count in pref_alimenticias.items():
    porcentaje = (count / len(df_restaurantes)) * 100
    print(f"   - {pref}: {count} restaurantes ({porcentaje:.1f}%)")

print("\n DISTRIBUCIoN DE CONSUMO DE LICOR:")
consume_licor = df_restaurantes['consume_licor'].value_counts()
for consumo, count in consume_licor.items():
    porcentaje = (count / len(df_restaurantes)) * 100
    print(f"   - {consumo}: {count} restaurantes ({porcentaje:.1f}%)")


# verificamos estructura acrualizada
print(f"\n ESTRUCTURA ACTUALIZADA DEL DATAFRAME:")
print(f"- Filas: {df_restaurantes.shape[0]}")
print(f"- Columnas: {df_restaurantes.shape[1]}")
print(f"- Nuevas columnas agregadas: 'preferencias_alimenticias', 'consume_licor'")



CLASIFICACION DE RESTAURANTES POR PREFERENCIAS
Agregando columnas...
CLASIFICACION COMPLETADA, LOS RESULTADOS SON:
------------------------------------------------------------
DISTRIBUCION DE PREFERENCIAS ALIMENTICIAS:
   - Otro: 141 restaurantes (70.5%)
   - Carnes: 25 restaurantes (12.5%)
   - Mariscos_Pescado: 21 restaurantes (10.5%)
   - Vegetariano_Vegano: 13 restaurantes (6.5%)

 DISTRIBUCIoN DE CONSUMO DE LICOR:
   - No: 107 restaurantes (53.5%)
   - Si: 93 restaurantes (46.5%)

 ESTRUCTURA ACTUALIZADA DEL DATAFRAME:
- Filas: 200
- Columnas: 21
- Nuevas columnas agregadas: 'preferencias_alimenticias', 'consume_licor'


In [21]:
#Revisemos si podemos crear un paralelismo entre estrato_socioeconomico (en el DF de clientes) y precio (en el DF de restaurantes) para generar insights interesantes

# 1. Exploremos Cliente
print("ESTRATO SOCIOECONOMICO EN CLIENTES:")
print("-" * 60)

if 'estrato_socioeconomico' in df_clientes.columns:
    # Ver valores unicos y distribucion
    estratos = df_clientes['estrato_socioeconomico'].value_counts().sort_index()
    print("Distribución de estratos socioeconómicos:")
    for estrato, count in estratos.items():
        porcentaje = (count / len(df_clientes)) * 100
        print(f"   - Estrato {estrato}: {count} clientes ({porcentaje:.1f}%)")
    
else:
    print("Columna 'estrato_socioeconomico' no encontrada en clientes")
    print("Columnas disponibles:", list(df_clientes.columns))

# 2. Exploremos Restaurantes
print("\n PRECIOS EN RESTAURANTES:")
print("-" * 60)

if 'precio' in df_restaurantes.columns:
    # Ver valores unicos y distribución
    precios = df_restaurantes['precio'].value_counts().sort_index()
    print("Distribución de precios en restaurantes:")
    for precio, count in precios.items():
        porcentaje = (count / len(df_restaurantes)) * 100
        print(f"   - {precio}: {count} restaurantes ({porcentaje:.1f}%)")
    
    # Verificar valores nulos
    nulos_precio = df_restaurantes['precio'].isnull().sum()
    print(f"   - Valores nulos: {nulos_precio}")
else:
    print("Columna 'precio' no encontrada en restaurantes")
    print("Columnas disponibles:", list(df_restaurantes.columns))


ESTRATO SOCIOECONOMICO EN CLIENTES:
------------------------------------------------------------
Distribución de estratos socioeconómicos:
   - Estrato Alto: 1501 clientes (29.6%)
   - Estrato Bajo: 874 clientes (17.2%)
   - Estrato Medio: 1775 clientes (35.0%)
   - Estrato Muy Alto: 919 clientes (18.1%)

 PRECIOS EN RESTAURANTES:
------------------------------------------------------------
Distribución de precios en restaurantes:
   - $$: 70 restaurantes (35.0%)
   - $$$: 48 restaurantes (24.0%)
   - $$$$: 12 restaurantes (6.0%)
   - No especificado: 70 restaurantes (35.0%)
   - Valores nulos: 0


### Paralelismo entre Precio y Estrato
#### Se establece un paralelismo entre estrato socioeconómico y precio de restaurantes, considerando solo los locales con precios definidos ($, $$, $$$, $$$$).
#### Los “No especificado” (31%) se excluyen en esta etapa y se analizarán luego en base a reseñas y nivel de actividad, para entender si su falta de precio se relaciona con baja visibilidad o información incompleta.

In [22]:
# Vamos a hacer un paralelismo entre Precio - estrato social
# 1. Armamos las equivalencias de estrato → precio (para analisis cruzado)
mapeo_precio_estrato = {
    '$$$$': 'Muy Alto',
    '$$$': 'Alto',
    '$$': 'Medio', 
    '$': 'Bajo',
    'No especificado': 'No especificado'
}
# 2. Cruzamos datos
print("Relacion Estrato Social del cliente con los precios de los Restaurantes:")
print("-" * 60)

# Invertir el mapeo completo a precio → estrato (para agregar columna a restaurantes)
mapeo_estrato_precio = {v: k for k, v in mapeo_precio_estrato.items()}

for estrato, precio in mapeo_estrato_precio.items():
    clientes = estratos.get(estrato, 0)
    restaurantes = precios.get(precio, 0)
    
    print(f"{estrato} → {precio}:")
    print(f"{clientes} clientes | {restaurantes} restaurantes")
   
# Ahora vamos a agregar una columna al dataset de Yelp, para recurrir a ella luego si es necesario
print("="*60)
print("AAgregando Estrato Socioeconomico Objetivo a df_restaurantes...")
print("="*60)

# Usar el mismo mapeo para la columna
df_restaurantes['estrato_socioeconomico'] = df_restaurantes['precio'].map(mapeo_precio_estrato)

# Mostrar distribucion
print("DISTRIBUCIÓN DE ESTRATO SOCIOECONÓMICO EN RESTAURANTES:")
print("-" * 60)

distribucion_estrato = df_restaurantes['estrato_socioeconomico'].value_counts()
for estrato, count in distribucion_estrato.items():
    porcentaje = (count / len(df_restaurantes)) * 100
    print(f"   - {estrato}: {count} restaurantes ({porcentaje:.1f}%)")

# Verificar la nueva columna
print(f"\n COLUMNA AGREGADA EXITOSAMENTE:")
print(f"- Dataset shape: {df_restaurantes.shape}")
print(f"- Nueva columna: 'estrato_socioeconomico'")
print(f"- Valores únicos: {df_restaurantes['estrato_socioeconomico'].nunique()}")

# Guardar dataset actualizado en la carpeta Archivos CSV
df_restaurantes.to_csv("../Archivos_CSV/restaurantes_yelp_con_estrato.csv", index=False)
print("Dataset actualizado guardado en: ../Archivos_CSV/restaurantes_yelp_con_estrato.csv")


Relacion Estrato Social del cliente con los precios de los Restaurantes:
------------------------------------------------------------
Muy Alto → $$$$:
919 clientes | 12 restaurantes
Alto → $$$:
1501 clientes | 48 restaurantes
Medio → $$:
1775 clientes | 70 restaurantes
Bajo → $:
874 clientes | 0 restaurantes
No especificado → No especificado:
0 clientes | 70 restaurantes
AAgregando Estrato Socioeconomico Objetivo a df_restaurantes...
DISTRIBUCIÓN DE ESTRATO SOCIOECONÓMICO EN RESTAURANTES:
------------------------------------------------------------
   - Medio: 70 restaurantes (35.0%)
   - No especificado: 70 restaurantes (35.0%)
   - Alto: 48 restaurantes (24.0%)
   - Muy Alto: 12 restaurantes (6.0%)

 COLUMNA AGREGADA EXITOSAMENTE:
- Dataset shape: (200, 22)
- Nueva columna: 'estrato_socioeconomico'
- Valores únicos: 4
Dataset actualizado guardado en: ../Archivos_CSV/restaurantes_yelp_con_estrato.csv


In [23]:
# Ahora revisemos en busca de oportunidades de los restaurantes con bajas reseñas

# 1. Vamos a buscar restaurantes con alto rating pero con pocas reseñas
umbral_alta_calidad = 4.0  # Restaurantes con rating >= 4.0
umbral_baja_visibilidad = 50  # Restaurantes con menos de 50 reseñas

oportunidades_trafico = df_restaurantes[
    (df_restaurantes['rating'] >= umbral_alta_calidad) & 
    (df_restaurantes['reseñas'] < umbral_baja_visibilidad)
].copy()

# Ordenar por potencial (rating alto + reseñas bajas)
oportunidades_trafico = oportunidades_trafico.sort_values(['rating', 'reseñas'], ascending=[False, True])

print(f"Se encontraron {len(oportunidades_trafico)} oportunidades de aumento de trafico")
print(f"- Alta calidad: rating {umbral_alta_calidad}+")
print(f"- Baja visibilidad: menos de {umbral_baja_visibilidad} reseñas")
print("-" * 60)

Se encontraron 31 oportunidades de aumento de trafico
- Alta calidad: rating 4.0+
- Baja visibilidad: menos de 50 reseñas
------------------------------------------------------------


In [24]:
# Sumemos el precio No Especifico a esta oportunidad
print(" OPORTUNIDADES - PRECIO NO ESPECIFICADO")
print("   Alto rating + baja visibilidad + precio por investigar")
print("="*60)

# Filtrar solo oportunidades con precio "No especificado"
oportunidades_no_especificado = oportunidades_trafico[
    oportunidades_trafico['precio'] == 'No especificado'
].copy()

# Calcular porcentaje correctamente
porcentaje_no_especificado = (len(oportunidades_no_especificado) / len(oportunidades_trafico)) * 100

print(f"Se encontraron {len(oportunidades_no_especificado)} oportunidades con precio no especificado")
print(f"  - Representan el {porcentaje_no_especificado:.1f}% de todas las oportunidades")
print(f"   - Rating ≥ {umbral_alta_calidad} | Reseñas < {umbral_baja_visibilidad}")
print("-" * 60)

# MOSTRAR TODAS LAS OPORTUNIDADES DE ESTE GRUPO
if len(oportunidades_no_especificado) > 0:
    print("\nTODAS LAS OPORTUNIDADES CON PRECIO NO ESPECIFICADO:")
    print("-" * 60)
    
    for i, (_, restaurante) in enumerate(oportunidades_no_especificado.iterrows(), 1):
        print(f"{i:2d}. {restaurante['nombre_negocio']}")
        print(f"   Rating: {restaurante['rating']} | Reseñas: {restaurante['reseñas']}")
        print(f"   Categorías: {restaurante['categorias']}")
        print()
#Vamos a guardar las oportunidades de rating y precios no especificos
if len(oportunidades_no_especificado) > 0:
    # Seleccionar solo las columnas requeridas
    columnas_guardar = [
        'nombre_negocio', 'rating', 'reseñas', 'categorias', 'precio',
        'direccion', 'telefono', 'url_yelp'
    ]
    
# Filtrar el DataFrame para mantener solo las columnas deseadas
oportunidades_guardar = oportunidades_no_especificado[columnas_guardar].copy()
    
# Guardar en CSV dentro de la carpeta Archivos CSV
oportunidades_guardar.to_csv("../Archivos_CSV/oportunidades_no_especificado.csv", index=False)
print("Archivo guardado en: ../Archivos_CSV/oportunidades_no_especificado.csv")
    
print(f"Archivo guardado con {len(oportunidades_guardar)} oportunidades")
print(f"Ruta: {ruta_guardado}")


 OPORTUNIDADES - PRECIO NO ESPECIFICADO
   Alto rating + baja visibilidad + precio por investigar
Se encontraron 30 oportunidades con precio no especificado
  - Representan el 96.8% de todas las oportunidades
   - Rating ≥ 4.0 | Reseñas < 50
------------------------------------------------------------

TODAS LAS OPORTUNIDADES CON PRECIO NO ESPECIFICADO:
------------------------------------------------------------
 1. Cariño
   Rating: 5.0 | Reseñas: 24
   Categorías: Latin American, Tacos

 2. Lao Der
   Rating: 5.0 | Reseñas: 34
   Categorías: Laotian

 3. To Korean Cuisine
   Rating: 4.9 | Reseñas: 12
   Categorías: Korean

 4. 3 Asian Sisters
   Rating: 4.9 | Reseñas: 22
   Categorías: Vietnamese

 5. Omakase Shoji
   Rating: 4.9 | Reseñas: 39
   Categorías: Japanese, Bars

 6. Pilot Project Brewing
   Rating: 4.8 | Reseñas: 6
   Categorías: Breweries, Cocktail Bars, Asian Fusion

 7. Matilda
   Rating: 4.8 | Reseñas: 24
   Categorías: Mexican, Peruvian, Cocktail Bars

 8. Mahanakho

# Continuamos en el Avance 3...